# グルモジ — Google Colab版

上から順にセルを実行すると、現在版の話者分離文字起こしWeb UIをColab内に表示します。OpenAI / Google APIに加えて、LM Studioの代わりにColab内のOllamaで動くローカルLLMを選択できます。

1. GPUを使う場合はColabの「ランタイム」→「ランタイムのタイプを変更」で **T4 GPU以上** を選択してください。GPUのない環境ではCPU向けの軽量LLMを選びます。
2. [Hugging Faceの既定の話者分離モデル](https://huggingface.co/pyannote/speaker-diarization-community-1)で利用条件に同意し、同じアカウントでread tokenを作成してください。
3. 左側の **鍵アイコン（シークレット）** に下記の名前とキーを登録し、**「ノートブックからのアクセス」をON** にしてください。キーをコードへ書く必要はありません。
4. 仕上げ用ローカルLLMはセル4の `LOCAL_LLM_MODEL = "auto"` で、ランタイムの空きGPU容量・RAMに応じて選びます。モデル名を入力すると直接指定を優先します。

既存のランタイムを更新する場合は、処理の停止・保存状態を確認してから終了セルを実行し、最新のノートブックを開き直してください。アプリの依存関係はColab標準環境と分けた専用Python環境へ導入します。セットアップには実行するGit版を表示し、ローカルLLMは短いJSON推論と実際の入力枠を確認してから準備完了にします。

> ColabのランタイムとGPUは保証されず、停止すると一時ファイルは消えます。出力を保持する場合はGoogle Drive保存を有効にしてください。ローカルLLMでは会話テキストは外部APIへ送信されませんが、初回はOllama公式配布元からプログラムとモデルをダウンロードします。Web UIは文字起こしという対話的計算のためだけに使用してください。

## 左側のシークレットでキーを設定する

「新しいシークレットを追加」で名前と値を登録します。値はセルの出力に表示しません。名前は大文字・小文字を含めて表どおりにしてください。

| シークレット名 | 必要な場面 | 発行・設定ページ |
| --- | --- | --- |
| `HF_TOKEN` | 話者分離に必要。利用条件に同意したアカウントのread token | [Hugging Faceのトークン作成](https://huggingface.co/settings/tokens)・[権限の説明](https://huggingface.co/docs/hub/security-tokens) |
| `OPENAI_API_KEY` | OpenAIを使う場合だけ | [OpenAIのAPIキー](https://platform.openai.com/api-keys) |
| `GOOGLE_API_KEY` | Geminiを使う場合だけ | [Google AI StudioのAPIキー](https://aistudio.google.com/apikey) |
| `TYPESAFE_API_KEY` | TypeSafe / JEVの校正を使う場合だけ | 利用中のTypeSafeサービスで発行したAPIキー |

`HUGGINGFACE_TOKEN` は `HF_TOKEN` の別名、`GEMINI_API_KEY` は `GOOGLE_API_KEY` の別名として使えます。シークレットが未登録の場合は、このランタイムの既存設定を保持します。キーを変更した場合は、セル2を再実行してアプリを停止・起動し直してください。シークレットを削除するだけでは既存設定のキーは消えません。利用を止める場合は提供元でキーを失効させ、ランタイムを作り直してください。

セル2は既定でHugging Faceの認証と、既定話者モデルのファイルへのアクセスを確認します。重みのダウンロードや推論は行いません。旧モデルを使う場合は [speaker-diarization-3.1](https://huggingface.co/pyannote/speaker-diarization-3.1) と [segmentation-3.0](https://huggingface.co/pyannote/segmentation-3.0) の利用条件も確認してください。fine-grained tokenを使う場合は、対象モデルを読み取れる権限が必要です。

OpenAI・Gemini・TypeSafeはキーの読込状態だけを表示し、有効性・残高・モデル利用権限は処理時に確認されます。設定はアプリへ渡すためランタイム内の `config/tokens.json` に保存します。Google Driveやノートブックの出力へキーを書き出すことはありません。シークレットが使えない場合だけ、セル2の `MANUAL_TOKEN_INPUT` をONにすると非表示の手入力を使えます。

## 仕上げ用LLMをランタイムに合わせる

セル4の `LOCAL_LLM_MODEL` は `auto`（空欄も同じ）が既定です。GPU名だけで決めず、現在の空きGPUメモリーと利用可能RAMから、音声処理用にGPU容量を4GiB以上残してモデルを選びます。長い本文・アウトラインと生成結果に使う枠は、自動では従来の32,768を維持します。T4 16GBではQwen3 4BのQ4、L4 24GBでは同Q8、A100 40GBではQwen3 30B-A3B InstructのQ4、80GB級では同Q8が目安です。他の処理でメモリーを使用中なら軽い候補になります。CPUではQwen2.5 1.5B〜3Bを選びます。複数GPUの容量は合算しません。

モデルを指定する場合は `LOCAL_LLM_MODEL` に `qwen3:4b-instruct`、`gemma3:4b`、`hf.co/組織/モデル:量子化タグ` など、Ollamaで取得できるローカルのテキストモデル名を入力してください。既定の候補一覧に制限しません。タグを省略すると `:latest` を補います。`LOCAL_LLM_CONTEXT_LENGTH` は0で自動、4096〜32768で直接指定できます。直接指定したモデルを、失敗時に別モデルへ変更することはありません。

選択理由・検出した容量・モデル名・入力枠を表示し、取得後に文章生成への対応・モデルの入力枠上限・JSON schema応答・実際の入力枠を検査します。資源からの推奨は品質の実測順位ではなく、入力内容に対する校正品質やGPUでの動作を保証するものではありません。JSON応答に対応しないモデルや容量不足は、理由を表示して停止します。Ollamaのクラウドモデルはこのローカル設定では使いません。

モデルの配布元: [Qwen3のタグとサイズ](https://ollama.com/library/qwen3/tags)・[Qwen2.5 3B](https://ollama.com/library/qwen2.5:3b-instruct)・[1.5B](https://ollama.com/library/qwen2.5:1.5b-instruct)。入力枠の考え方は [Ollama公式の説明](https://docs.ollama.com/context-length) を参照してください。

## エラーが表示されたら

| 表示される分類 | 主な原因と対処 |
| --- | --- |
| `SECRET_MISSING` | 左側に `HF_TOKEN` を追加し、アクセスをONにしてセル2を再実行 |
| `SECRET_ACCESS` | 指定されたシークレットの「ノートブックからのアクセス」をONにする |
| `SECRET_TIMEOUT` / `SECRET_READ` | Colabの画面との接続とシークレット設定を確認してセル2を再実行 |
| `TOKEN_FILE` / `TOKEN_WRITE` | 既存設定を保全し、JSON形式・書込権限・空き容量を確認 |
| `AUTH_401` | キーの失効・誤入力を確認し、左側の値を更新してセル2を再実行 |
| `PERMISSION_403` | 同じアカウントでモデルの利用条件に同意し、キーのread権限・対象モデル・APIのプロジェクトと提供地域を確認 |
| `MODEL_404` / `REQUEST_400` | モデル名・URL・要求形式・モデルへのアクセス権を確認 |
| `LIMIT_429` | 回数制限なら待機。残高・利用枠不足なら提供元の利用状況や課金設定を確認 |
| `NETWORK` / `TLS` / `TIMEOUT` / `SERVICE_5XX` | 通信・証明書・提供元の稼働状況を確認し、時間をおいて再実行 |
| `ENVIRONMENT` / `DISK_FULL` | apt/pipの直前の出力・空き容量を確認。旧版で標準環境が変わった場合は、結果を保存してランタイムを再作成 |
| `GPU_RUNTIME` / `GPU_MEMORY` | GPUへの接続を確認。小さいモデル・短いコンテキストを選ぶか、任意のローカルLLMをOFFにする |
| `DRIVE` | GoogleアカウントとDriveのアクセス許可を確認。DriveをOFFにすると一時保存 |
| `LLM_MODEL` / `LLM_CONTEXT` / `LLM_MEMORY` | モデル名・入力枠を確認。RAM不足は容量を確保するかローカルLLMをOFFにする |
| `LOCAL_LLM` | モデル名・通信・GPU容量を確認してセル4を再実行。利用しない場合は `USE_LOCAL_LLM` をOFF |
| `HF_CHECK` / `HF_CHECK_FAILED` | 直前のHugging Face診断と上記のモデルページを確認。`CHECK_HF_ACCESS` をOFFにすると事前確認を省略するだけで、権限は解決されない |
| `APP_START` | セル1〜4の完了と既存プロセスの停止を確認し、セル5を再実行 |

外部AIが実行中に返すHTTPエラーも上記の400・401・403・404・429・5xxの説明で確認できます。APIの詳細は [OpenAI公式のエラー説明](https://developers.openai.com/api/docs/guides/error-codes)・[Gemini公式のエラー説明](https://ai.google.dev/gemini-api/docs/api-errors) を参照してください。報告時は発生セルと分類を添え、キー・録音内容は含めないでください。


## 任意：下部にランタイムの使用状況を表示する

Web UIの下にあるセル6で `ENABLE_RUNTIME_MONITOR` をONにして実行すると、CPU使用率、GPU使用率とGPUメモリー、RAMの使用量を表示します。既定はOFF、更新間隔は2秒（1〜30秒で変更可）です。直近60回の使用率をグラフにし、値を取得できない場合は「未取得」と表示します。GPUがない環境でもCPUとRAMは表示できます。

計測はバックグラウンドで動き、セルを実行し続ける必要はありません。「モニターを停止」ボタン、OFFにしてセル6を再実行、または終了セル7で停止します。セル6を再実行すると前の計測を止めてから表示し直します。計測はこのColabランタイム全体が対象で、文字起こしやLLMだけの使用率ではありません。履歴はメモリー内だけに保持し、キー・録音内容は読み取りません。


In [ ]:
# 1. 現在版のリポジトリと依存関係を準備（初回は数分かかります）
from pathlib import Path
import hashlib
import os
import subprocess
import sys
import venv

try:
    import google.colab  # noqa: F401
except ImportError as exc:
    raise RuntimeError("このノートブックはGoogle Colabで実行してください。") from exc

REPO_URL = "https://github.com/krokharu/Gurumoji-public.git"
REPO_DIR = Path("/content/gurumoji")

def prepare_colab_repository(repo_dir, repo_url, processes=()):
    if any(process is not None and process.poll() is None for process in processes):
        raise RuntimeError("アプリ/Ollamaが実行中です。処理の停止・保存状態を確認し、終了セルを実行してから更新してください。")
    if (repo_dir / ".git").exists():
        branch = subprocess.check_output(["git", "-C", str(repo_dir), "branch", "--show-current"], text=True).strip()
        changes = subprocess.check_output(["git", "-C", str(repo_dir), "status", "--porcelain", "--untracked-files=no"], text=True).strip()
        if branch != "public/release" or changes:
            raise RuntimeError("public/release以外の版またはローカル編集があります。変更を保全してから更新してください。自動上書きはしません。")
        subprocess.run(["git", "-C", str(repo_dir), "pull", "--ff-only", "origin", "public/release"], check=True)
    else:
        subprocess.run(["git", "clone", "--branch", "public/release", "--depth", "1", repo_url, str(repo_dir)], check=True)
    return subprocess.check_output(["git", "-C", str(repo_dir), "rev-parse", "HEAD"], text=True).strip()

GURUMOJI_COMMIT = prepare_colab_repository(
    REPO_DIR, REPO_URL, (globals().get("gurumoji_server"), globals().get("ollama_server")))
print("使用するGurumojiの版:", GURUMOJI_COMMIT)

sys.path.insert(0, str(REPO_DIR / "src"))
from gurumoji.colab_support import ColabSetupError, colab_step, load_colab_tokens

with colab_step("ENVIRONMENT"):
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg", "fonts-noto-cjk", "zstd"], check=True)

def prepare_colab_environment(repo_dir, environment_root=Path("/content/gurumoji-envs")):
    # Colabの標準パッケージと音声処理の依存関係を混在させない。
    requirements = (repo_dir / "config" / "requirements.txt").read_bytes()
    setup_hash = hashlib.sha256(
        requirements + sys.version.encode() + b"colab-cu128-venv-v1"
    ).hexdigest()[:12]
    environment_dir = environment_root / setup_hash
    colab_python = environment_dir / "bin" / "python"
    setup_marker = environment_dir / ".setup-complete"
    ready = setup_marker.is_file() and colab_python.is_file()
    setup_marker.unlink(missing_ok=True)
    if not ready:
        venv.EnvBuilder(with_pip=False, system_site_packages=False).create(environment_dir)
        # Colab側のpipから専用環境へ導入し、ensurepipの有無に依存しない。
        subprocess.run([
            sys.executable, "-m", "pip", "--python", str(colab_python),
            "install", "--upgrade", "pip",
        ], check=True)
        subprocess.run([
            str(colab_python), "-m", "pip", "install", "--no-cache-dir",
            "torch==2.8.0+cu128", "torchvision==0.23.0+cu128", "torchaudio==2.8.0+cu128",
            "--index-url", "https://download.pytorch.org/whl/cu128",
        ], check=True)
        subprocess.run([
            str(colab_python), "-m", "pip", "install", "--no-cache-dir",
            "-r", str(repo_dir / "config" / "requirements.txt"),
        ], check=True)
    # 再実行時も専用環境を検査し、失敗した準備を完了扱いにしない。
    subprocess.run([str(colab_python), "-m", "pip", "check"], check=True)
    setup_marker.touch()
    return colab_python

with colab_step("ENVIRONMENT"):
    COLAB_PYTHON = prepare_colab_environment(REPO_DIR)
print("アプリ専用Python:", COLAB_PYTHON)

os.chdir(REPO_DIR)
print("準備完了:", REPO_DIR)


In [ ]:
# 2. 左側のシークレットからAPIキーを読み込み、Hugging Faceを確認
from google.colab import userdata
from getpass import getpass
import json

MANUAL_TOKEN_INPUT = False  # @param {type:"boolean"}
CHECK_HF_ACCESS = True  # @param {type:"boolean"}

with colab_step("SECRETS"):
    token_path = REPO_DIR / "config" / "tokens.json"
    tokens, token_sources = load_colab_tokens(
        token_path, userdata, manual_input=getpass if MANUAL_TOKEN_INPUT else None)
    for name, source in token_sources.items():
        print(f"{name}: {source}（値は非表示）")
    print("接続設定をランタイム内に保存しました。外部AIの有効性・利用枠は未確認です。")

if CHECK_HF_ACCESS:
    hf_check_env = os.environ.copy()
    hf_check_env["PYTHONPATH"] = str(REPO_DIR / "src")
    hf_check = subprocess.run(
        [str(COLAB_PYTHON), "-m", "gurumoji.colab_support", str(token_path)],
        cwd=str(REPO_DIR), env=hf_check_env)
    if hf_check.returncode:
        raise ColabSetupError("HF_CHECK_FAILED", "直前の認証・モデル利用条件の案内を確認し、セル2を再実行してください。")
else:
    print("Hugging Faceへのアクセスは未確認です。話者分離の実行時に認証・モデル取得が必要です。")


In [ ]:
# 3. 出力・話者管理データをGoogle Driveへ保持するか選択
USE_GOOGLE_DRIVE = True  # @param {type:"boolean"}

with colab_step("DRIVE"):
    if USE_GOOGLE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        storage_root = Path("/content/drive/MyDrive/gurumoji")
    else:
        storage_root = REPO_DIR / "runtime"

    output_dir = storage_root / "output"
    data_dir = storage_root / "data"
    output_dir.mkdir(parents=True, exist_ok=True)
    data_dir.mkdir(parents=True, exist_ok=True)
    os.environ["MOJIOKOSI_OUTPUT_DIR"] = str(output_dir)
    os.environ["MOJIOKOSI_DATA_DIR"] = str(data_dir)
    print("出力先:", output_dir)
    print("話者管理・ライブラリ:", data_dir)


In [ ]:
# 4. ColabローカルLLM（LM Studioの代わり）を選択・起動
USE_LOCAL_LLM = True  # @param {type:"boolean"}
LOCAL_LLM_MODEL = "auto"  # @param {type:"string"}
LOCAL_LLM_CONTEXT_LENGTH = 0  # @param {type:"integer"}
PERSIST_LOCAL_LLM_MODEL = False  # @param {type:"boolean"}
LOCAL_LLM_READY = None

import shutil
import time
import urllib.request
import urllib.error

from gurumoji.colab_llm import detect_colab_llm_runtime, select_colab_llm, ollama_model_matches

def ollama_json(path, payload=None, timeout=10):
    request = urllib.request.Request(
        "http://127.0.0.1:11434" + path,
        data=json.dumps(payload).encode("utf-8") if payload is not None else None,
        headers={"Content-Type": "application/json"},
    )
    try:
        with urllib.request.urlopen(request, timeout=timeout) as response:
            return json.load(response)
    except urllib.error.HTTPError as exc:
        if exc.code in {400, 500}:
            # ローカル応答は分類だけに使い、例外本文は再表示しない。
            detail = exc.read(4096).decode("utf-8", errors="replace").lower()
            if any(text in detail for text in ("out of memory", "more system memory", "not enough memory", "unable to allocate")):
                raise ColabSetupError("LLM_MEMORY", "モデルを読み込むメモリーが不足しています。小さいモデルや短い入力枠を指定するか、ローカルLLMをOFFにしてください。") from None
        raise

def verify_colab_local_llm(model, context_length):
    information = ollama_json("/api/show", {"model": model})
    capabilities = information.get("capabilities")
    if isinstance(capabilities, list) and "completion" not in capabilities:
        raise ColabSetupError("LLM_MODEL", "指定モデルは文章生成に対応していません。JSON応答に対応したテキストモデルを選んでください。")
    model_info = information.get("model_info", {})
    architecture = model_info.get("general.architecture")
    maximum_context = model_info.get(f"{architecture}.context_length")
    if type(maximum_context) is int and maximum_context > 0 and context_length > maximum_context:
        raise ColabSetupError("LLM_CONTEXT", f"指定した入力枠はモデルの上限 {maximum_context} を超えています。LOCAL_LLM_CONTEXT_LENGTHをモデルの範囲内に設定してください。")
    # 架空の短い入力で、アプリと同じJSON schema形式を確認する。
    reply = ollama_json("/v1/chat/completions", {
        "model": model, "stream": False, "temperature": 0, "reasoning_effort": "none",
        "messages": [{"role": "user", "content": 'Return only the JSON object {"status":"ready"}.'}],
        "response_format": {"type": "json_schema", "json_schema": {
            "name": "gurumoji_startup", "strict": True, "schema": {
                "type": "object", "properties": {"status": {"type": "string", "enum": ["ready"]}},
                "required": ["status"], "additionalProperties": False,
            }}},
        "max_tokens": 64,
    }, timeout=180)
    try:
        choice = reply["choices"][0]
        if choice.get("finish_reason") != "stop" or json.loads(choice["message"]["content"]) != {"status": "ready"}:
            raise ValueError("incomplete or invalid startup response")
    except (KeyError, IndexError, TypeError, ValueError) as exc:
        raise RuntimeError("ローカルLLMの短いJSON推論を確認できませんでした。モデルとOllamaのログを確認してください。") from exc
    running = ollama_json("/api/ps")
    loaded = next((entry for entry in running.get("models", [])
                   if ollama_model_matches(entry.get("name"), model) or ollama_model_matches(entry.get("model"), model)), None)
    if loaded is None:
        raise RuntimeError("推論後のロード済みモデルを確認できませんでした。")
    actual_context = loaded.get("context_length")
    if type(actual_context) is not int or actual_context < context_length:
        raise RuntimeError("実際の入力枠が指定値より小さいか未取得です。Ollamaとコンテキスト設定を確認してください。")
    return {"model": model, "context_length": actual_context, "size_vram": loaded.get("size_vram")}

with colab_step("LOCAL_LLM"):
    if "ollama_server" in globals() and ollama_server.poll() is None:
        ollama_server.terminate()
        ollama_server.wait(timeout=15)

    if USE_LOCAL_LLM:
        local_llm_runtime = detect_colab_llm_runtime()
        local_llm_plan = select_colab_llm(local_llm_runtime, LOCAL_LLM_MODEL, LOCAL_LLM_CONTEXT_LENGTH)
        LOCAL_LLM_MODEL = local_llm_plan["model"]
        LOCAL_LLM_CONTEXT_LENGTH = local_llm_plan["context_length"]
        selection_mode = "自動選択" if local_llm_plan["automatic"] else "直接指定"
        print(f"仕上げ用LLM（{selection_mode}）: {LOCAL_LLM_MODEL} / 入力枠 {LOCAL_LLM_CONTEXT_LENGTH}")
        print(local_llm_plan["reason"])
        if local_llm_plan["gpu_name"]:
            print(f"GPU: {local_llm_plan['gpu_name']} / 空き {local_llm_plan['gpu_free_gib']:.1f} GiB / 音声処理用の余裕 {local_llm_plan['gpu_reserve_gib']:.1f} GiB")
        if local_llm_plan["ram_available_gib"] is not None:
            print(f"利用可能RAM: {local_llm_plan['ram_available_gib']:.1f} GiB")
        if shutil.which("ollama") is None:
            archive_path = Path("/content/ollama-linux-amd64.tar.zst")
            print("Ollamaを公式配布元からダウンロードしています…")
            urllib.request.urlretrieve(
                "https://ollama.com/download/ollama-linux-amd64.tar.zst",
                archive_path,
            )
            subprocess.run(["tar", "--zstd", "-xf", str(archive_path), "-C", "/usr"], check=True)
            archive_path.unlink(missing_ok=True)

        ollama_models = (storage_root / "models" / "ollama") if PERSIST_LOCAL_LLM_MODEL else Path("/content/ollama-models")
        ollama_models.mkdir(parents=True, exist_ok=True)
        ollama_env = os.environ.copy()
        ollama_env.update({
            "OLLAMA_HOST": "127.0.0.1:11434",
            "OLLAMA_NO_CLOUD": "1",
            "OLLAMA_MODELS": str(ollama_models),
            "OLLAMA_CONTEXT_LENGTH": str(int(LOCAL_LLM_CONTEXT_LENGTH)),
            "OLLAMA_NUM_PARALLEL": "1",
            "OLLAMA_FLASH_ATTENTION": "1",
            "OLLAMA_KV_CACHE_TYPE": "q8_0",
            "OLLAMA_KEEP_ALIVE": "5m",
        })
        if local_llm_plan["cuda_visible_devices"] is not None:
            ollama_env["CUDA_VISIBLE_DEVICES"] = local_llm_plan["cuda_visible_devices"]
        ollama_log_path = Path("/content/gurumoji_ollama.log")
        ollama_log = ollama_log_path.open("w", encoding="utf-8")
        ollama_server = subprocess.Popen(
            ["ollama", "serve"],
            env=ollama_env,
            stdout=ollama_log,
            stderr=subprocess.STDOUT,
        )
        for _ in range(60):
            if ollama_server.poll() is not None:
                ollama_log.flush()
                raise RuntimeError(ollama_log_path.read_text(encoding="utf-8", errors="replace")[-5000:])
            try:
                with urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2):
                    break
            except Exception:
                time.sleep(1)
        else:
            raise TimeoutError("Ollamaの起動がタイムアウトしました。/content/gurumoji_ollama.log を確認してください。")

        print(f"ローカルLLM {LOCAL_LLM_MODEL} を取得しています…")
        subprocess.run(["ollama", "pull", LOCAL_LLM_MODEL], env=ollama_env, check=True)
        with urllib.request.urlopen("http://127.0.0.1:11434/v1/models", timeout=10) as response:
            model_catalog = json.loads(response.read().decode("utf-8"))
        available_ids = {item.get("id") for item in model_catalog.get("data", [])}
        if not any(ollama_model_matches(model_id, LOCAL_LLM_MODEL) for model_id in available_ids):
            raise RuntimeError("取得したローカルLLMをOpenAI互換APIで確認できませんでした。")
        print("短いJSON推論を確認しています（初回はモデルのロードに時間がかかります）…")
        LOCAL_LLM_READY = verify_colab_local_llm(LOCAL_LLM_MODEL, LOCAL_LLM_CONTEXT_LENGTH)
        tokens.update({
            "lmstudio_base_url": "http://127.0.0.1:11434/v1",
            "lmstudio_model": LOCAL_LLM_MODEL,
            "lmstudio_api_key": "",
        })
        token_path.write_text(json.dumps(tokens, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
        vram = LOCAL_LLM_READY["size_vram"]
        gpu_report = f"GPU配置 {vram / 1024**3:.2f} GiB" if type(vram) is int and vram > 0 else "CPUのみ" if vram == 0 and type(vram) is int else "GPU配置未取得"
        print(f"JSON推論確認済み: {LOCAL_LLM_MODEL} / 実入力枠 {LOCAL_LLM_READY['context_length']} / {gpu_report}")
        if type(vram) is int and vram == 0:
            print("GPU推論は未確認です。ランタイムのGPU選択とOllamaのログを確認してください。")
    else:
        tokens.update({"lmstudio_base_url": "http://127.0.0.1:11434/v1", "lmstudio_model": ""})
        token_path.write_text(json.dumps(tokens, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
        print("ColabローカルLLMは使用しません。OpenAI / Googleは引き続き選択できます。")


In [ ]:
# 5. 現在版Web UIを起動してColab内に表示
from google.colab import output
import time
import urllib.request

with colab_step("APP_START"):
    PORT = 7860
    if USE_LOCAL_LLM and (not LOCAL_LLM_READY or LOCAL_LLM_READY["model"] != LOCAL_LLM_MODEL):
        raise RuntimeError("先にColabローカルLLMセルのJSON推論確認を完了してください。")
    if "gurumoji_server" in globals() and gurumoji_server.poll() is None:
        gurumoji_server.terminate()
        gurumoji_server.wait(timeout=15)

    server_log_path = Path("/content/gurumoji_server.log")
    server_log = server_log_path.open("w", encoding="utf-8")
    server_env = os.environ.copy()
    server_env.update({
        "PYTHONPATH": str(REPO_DIR / "src"),
        "MOJIOKOSI_RUNTIME": "colab",
        "MOJIOKOSI_NO_BROWSER": "1",
        "MOJIOKOSI_HOST": "127.0.0.1",
        "MOJIOKOSI_PORT": str(PORT),
    })
    gurumoji_server = subprocess.Popen(
        [str(COLAB_PYTHON), "-m", "gurumoji.app"],
        cwd=str(REPO_DIR),
        env=server_env,
        stdout=server_log,
        stderr=subprocess.STDOUT,
    )

    for _ in range(90):
        if gurumoji_server.poll() is not None:
            server_log.flush()
            raise RuntimeError(server_log_path.read_text(encoding="utf-8", errors="replace")[-5000:])
        try:
            with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/", timeout=2) as response:
                if response.status == 200:
                    break
        except Exception:
            time.sleep(1)
    else:
        raise TimeoutError("Web UIの起動がタイムアウトしました。/content/gurumoji_server.log を確認してください。")

    print("使用するGurumojiの版:", GURUMOJI_COMMIT)
    print("グルモジを起動しました。下の画面からファイルをアップロードできます。")
    if USE_LOCAL_LLM:
        print(f"AI仕上げ: Ollama（ColabローカルLLM）/ {LOCAL_LLM_MODEL}")
    output.serve_kernel_port_as_iframe(PORT, height=1100)


In [ ]:
# 6. 任意：ランタイムのCPU・GPU・RAMを表示
ENABLE_RUNTIME_MONITOR = False  # @param {type:"boolean"}
MONITOR_INTERVAL_SECONDS = 2  # @param {type:"integer"}

from gurumoji.colab_monitor import start_runtime_monitor

previous_monitor = globals().get("gurumoji_runtime_monitor")
if previous_monitor is not None:
    previous_monitor.stop()
gurumoji_runtime_monitor = None

if ENABLE_RUNTIME_MONITOR:
    gurumoji_runtime_monitor = start_runtime_monitor(
        COLAB_PYTHON, REPO_DIR, interval=MONITOR_INTERVAL_SECONDS)
    print("表示を開始しました。停止ボタン、またはOFFにしてセル6を再実行すると停止します。")
else:
    print("ランタイムモニターはOFFです。表示する場合だけONにしてこのセルを実行してください。")


In [ ]:
# 7. 終了するときだけ実行
stopped = []
if globals().get("gurumoji_runtime_monitor") is not None:
    gurumoji_runtime_monitor.stop()
    stopped.append("ランタイムモニター")
if "gurumoji_server" in globals() and gurumoji_server.poll() is None:
    gurumoji_server.terminate()
    gurumoji_server.wait(timeout=15)
    stopped.append("グルモジ")
if "ollama_server" in globals() and ollama_server.poll() is None:
    ollama_server.terminate()
    ollama_server.wait(timeout=15)
    stopped.append("Ollama")
print(" / ".join(stopped) + " を停止しました。" if stopped else "すでに停止しています。")
